# 02 — RWKV Architecture Lab

Prototype and explore the **RWKV** architecture using the building blocks in
`src/core/`.  This is where you iterate before promoting to `src/`.

In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
from src.models.rwkv.config import RWKVConfig
from src.models.rwkv.model import RWKVSLM
from src.core.generation import generate
from src.utils.training import count_params

In [ ]:
# ── Build a tiny model for fast iteration ────────────────────────────────
tiny_cfg = RWKVConfig(
    vocab_size=50257, block_size=32,
    n_layer=2, n_embd=64, n_head=2,
    ffn_mult=4, dropout=0.0,
)
model = RWKVSLM(tiny_cfg)
print(f'Parameters: {count_params(model):,}')

In [ ]:
# ── Verify forward pass shapes ────────────────────────────────────────────
B, T = 2, tiny_cfg.block_size
idx = torch.randint(0, tiny_cfg.vocab_size, (B, T))
targets = torch.randint(0, tiny_cfg.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f'logits: {logits.shape}  |  loss: {loss.item():.4f}')

In [ ]:
# ── Test generation ───────────────────────────────────────────────────────
prompt = torch.randint(0, tiny_cfg.vocab_size, (1, 4))
output = generate(model, prompt, max_new_tokens=10, temperature=1.0, top_k=50)
print(f'Input tokens:  {prompt[0].tolist()}')
print(f'Output tokens: {output[0].tolist()}')

In [ ]:
# ── Parameter count comparison across model sizes ────────────────────────
configs = {
    'rwkv_tiny':   RWKVConfig(vocab_size=50257, block_size=16,  n_layer=2,  n_embd=64,  n_head=2,  ffn_mult=4),
    'rwkv_small':  RWKVConfig(vocab_size=50257, block_size=128, n_layer=8,  n_embd=512, n_head=8,  ffn_mult=4),
    'rwkv_medium': RWKVConfig(vocab_size=50257, block_size=256, n_layer=16, n_embd=768, n_head=12, ffn_mult=4),
}
for name, cfg in configs.items():
    n = count_params(RWKVSLM(cfg))
    print(f'{name:14s}: {n:>12,} params  ({n/1e6:.1f}M)')